In [2]:
!c++ -O3 -mavx2 -mfma -fopenmp -shared -std=c++17 -fPIC \
!(python -m pybind11 --includes) \
!binding.cpp matmul.cpp \
!-o fastgemm$(python3-config --extension-suffix)
!/home/codespace/.python/current/bin/python: No module named pybind11

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `c++ -O3 -mavx2 -mfma -fopenmp -shared -std=c++17 -fPIC  !(python -m pybind11 --includes)  !binding.cpp matmul.cpp  !-o fastgemm$(python3-config --extension-suffix)'


/bin/bash: line 1: /home/codespace/.python/current/bin/python:: No such file or directory


In [3]:
!pip install numpy torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 16.9 MB/s  0:00:20m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 9.7 MB/s  0:00:0136m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 30.2 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 72.2 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 63.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 20.9 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 50.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 69.8 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 54.2 MB/

In [7]:
!pip install pybind11


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
import os
print(os.getcwd())

/workspaces/matrix_multiplication/fast_gemm


In [1]:
import pybind11
from setuptools import setup, Extension
from setuptools.command.build_ext import build_ext
import sys

ext_modules = [
    Extension(
        "fastgemm",
        ["binding.cpp", "matmul.cpp"],
        include_dirs=[pybind11.get_include()],
        extra_compile_args=["-O3", "-mavx2", "-mfma", "-fopenmp"],
        extra_link_args=["-fopenmp"],
        language="c++"
    )
]

setup(
    name="fastgemm",
    ext_modules=ext_modules,
    cmdclass={"build_ext": build_ext},
    script_args=["build_ext", "--inplace"],
)

running build_ext
building 'fastgemm' extension
g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O3 -Wall -fPIC -I/usr/local/python/3.12.1/lib/python3.12/site-packages/pybind11/include -I/usr/local/python/3.12.1/include/python3.12 -c binding.cpp -o build/temp.linux-x86_64-cpython-312/binding.o -O3 -mavx2 -mfma -fopenmp
g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O3 -Wall -fPIC -I/usr/local/python/3.12.1/lib/python3.12/site-packages/pybind11/include -I/usr/local/python/3.12.1/include/python3.12 -c matmul.cpp -o build/temp.linux-x86_64-cpython-312/matmul.o -O3 -mavx2 -mfma -fopenmp
creating build/lib.linux-x86_64-cpython-312
g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O3 -Wall -shared build/temp.linux-x86_64-cpython-312/binding.o build/temp.linux-x86_64-cpython-312/matmul.o -o build/lib.linux-x86_64-cpython-312/fastgemm.cpython-312-x86_64-linux-gnu.so -fopenmp
copying build/lib.linux-x86_64-cpython-312/fastgemm.cpython-312-x86_64-linux-gnu.so -> 


In [3]:
import sys
sys.path.append("/workspaces/matrix_multiplication/fast_gemm")

import fastgemm

In [10]:
import numpy as np

A = np.random.rand(128,128)
B = np.random.rand(128,128)

C = fastgemm.matmul(A,B)

print(C[:2,:2])

[[29.92265093 33.42050167]
 [27.14656477 30.67407589]]


In [13]:
N = 256

A = np.random.rand(N,N)
B = np.random.rand(N,N)

C1 = fastgemm.matmul(A,B)
C2 = A @ B

print("max error:", np.max(np.abs(C1 - C2)))

max error: 15.089034830323108


In [15]:
import time
N = 1024

A = np.random.rand(N,N)
B = np.random.rand(N,N)

# warmup
fastgemm.matmul(A,B)

start = time.time()
C = fastgemm.matmul(A,B)
end = time.time()

print("C++ GEMM:", end-start)

C++ GEMM: 0.11819601058959961


In [17]:
import torch
A_t = torch.from_numpy(A)
B_t = torch.from_numpy(B)

# CPU
start = time.time()
C_t = A_t @ B_t
end = time.time()

print("PyTorch CPU:", end-start)

PyTorch CPU: 0.10888504981994629
